# Experiment 1: full case corpus

One DAMICORE object represents one eligible category, while each document preserves
the canonical contextual combinations observed in all distinct `source_hash + category`
cases. Category prevalence is retained.


In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hypotheses.violence_against_women.scripts.experiment_common import (
    artifact_paths,
    case_execution,
    category_order_from_manifest,
    ensure_artifact_directories,
    load_artifact_manifest,
    load_category_map,
    run_damicore_experiment,
    write_common_result_artifacts,
)

CATEGORY_SET_VERSION = os.getenv("DAMICORE_CATEGORY_SET_VERSION", "v2_30")
PATHS = artifact_paths(CATEGORY_SET_VERSION)
COMMON_WORK_ROOT = PATHS.common
NORMALIZED_WORK_ROOT = PATHS.normalized
CASE_FULL_WORK_ROOT = PATHS.case_full
CASE_BALANCED_WORK_ROOT = PATHS.case_balanced
RESULTS_ROOT = PATHS.results
ensure_artifact_directories(PATHS)
manifest = load_artifact_manifest(
    COMMON_WORK_ROOT / "artifact-manifest.json",
    category_set_version=CATEGORY_SET_VERSION,
)
category_order = category_order_from_manifest(manifest)
category_map = load_category_map(COMMON_WORK_ROOT / "category-map.csv")
category_support = pd.read_csv(COMMON_WORK_ROOT / "category-support.csv")
assert category_map["category"].tolist() == category_order
assert set(category_support["category"]) == set(category_order)
assert manifest["dimension_count"] == 20


/Users/erickpatrickbarcelos/codes/data-mining/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [Errno 2] No such file or directory: '/Users/erickpatrickbarcelos/codes/data-mining/hypotheses/violence_against_women/artifacts/versions/v2_30/work/common/artifact-manifest.json'

## Run DAMICORE and record the common result contract


In [ ]:
case_category_map = pd.read_csv(COMMON_WORK_ROOT / "case-category-map.csv")
case_full_support = (
    case_category_map.loc[case_category_map["regime"] == "case-full", ["category", "case_count"]]
    .rename(columns={"case_count": "support"})
    .set_index("category")
    .loc[category_order]
    .reset_index()
)
bytes_by_category = (
    case_category_map.loc[case_category_map["regime"] == "case-full"]
    .set_index("category")["bytes"]
    .astype(int)
    .to_dict()
)
assert case_full_support["category"].tolist() == category_order

result = run_damicore_experiment(
    experiment_name="case_full",
    corpus_dir=CASE_FULL_WORK_ROOT / "corpus",
    raw_runs_dir=CASE_FULL_WORK_ROOT / "runs",
    execution=case_execution(),
)
assert result["status"] == "completed", result["preview"]
case_full_result = write_common_result_artifacts(
    result=result,
    category_map=category_map,
    category_order=category_order,
    support=case_full_support,
    bytes_by_category=bytes_by_category,
    output_dir=RESULTS_ROOT / "case_full",
    support_column="support",
    support_label="Distinct cases (log scale)",
    title_prefix="Case-full",
    manifest=manifest,
)
display(case_full_result["membership_by_category"])
display(case_full_result["distance"].round(3))


## Interpretation boundary

`case-full` combines contextual similarity with observed category prevalence and
corpus volume. It is not directly comparable to an equal-support regime without the
balanced experiment.
